# Lesson 16 — Convex Relaxations

## Learning objectives

1. Build the McCormick envelope of a bilinear term {cite:p}`McCormick1976`.
2. Recognize secant relaxations of univariate concave functions.
3. State the αBB convex underestimator {cite:p}`Adjiman1998,Adjiman1998a`.
4. Use `discopt`'s convex-relaxation infrastructure on a small NLP.

## 1. The McCormick envelope

For a bilinear term $z = x y$ with $x \in [\underline x, \overline x], y \in [\underline y, \overline y]$, the **convex hull** is the four McCormick inequalities:

$$
\begin{aligned}
z &\ge \underline x \, y + x \, \underline y - \underline x \, \underline y \\
z &\ge \overline x \, y + x \, \overline y - \overline x \, \overline y \\
z &\le \underline x \, y + x \, \overline y - \underline x \, \overline y \\
z &\le \overline x \, y + x \, \underline y - \overline x \, \underline y
\end{aligned}
$$

These give the *tightest* convex relaxation of the graph $\{(x, y, z) : z = xy\}$ over the box. Bound width determines tightness — narrower boxes → tighter envelopes.

## 2. Secant relaxation for concave $f$

For univariate concave $f$ on $[a, b]$:

- Tightest **convex underestimator**: the secant $f(a) + (f(b) - f(a))(x - a)/(b - a)$.
- Tightest **concave overestimator**: $f$ itself.

For convex $f$, swap the roles. Combined with McCormick for products, this handles "factorable" nonconvex problems {cite:p}`McCormick1976,Smith1999`.

## 3. αBB underestimator for general C² functions

For any $f \in C^2$ on a box, the αBB underestimator is

$$\check f(x) = f(x) - \sum_i \alpha_i (\overline x_i - x_i)(x_i - \underline x_i)$$

with $\alpha_i$ large enough to cancel any negative curvature of $f$. Explicit choices use eigenvalue bounds on $\nabla^2 f$ {cite:p}`Adjiman1998`. Convex by construction; tightness depends on $\alpha$.

This is the workhorse for global optimization of general factorable NLPs {cite:p}`Adjiman1998a,Tawarmalani2005`.

In [ ]:
import numpy as np, discopt as do
from discopt.modeling import Model

# Bilinear: min x*y  s.t.  x + y >= 1, x,y in [0, 2]
m = Model("bilinear")
x = m.add_variable(lb=0, ub=2)
y = m.add_variable(lb=0, ub=2)
z = m.add_variable(lb=-10, ub=10)            # auxiliary for the bilinear
m.add_constraint(z == x * y)
m.add_constraint(x + y >= 1)
m.minimize(z)

r_local  = m.solve(mode="local",  x0=[1.0, 1.0])
r_global = m.solve(mode="global")
print(f"local : z = {r_local.objective:.4f}")
print(f"global: z = {r_global.objective:.4f}")

# Inspect the McCormick envelope
print("McC inequalities at root node:", m.relaxation_inequalities())


## 4. Tightness vs branching

The McCormick relaxation is exact at the box corners; loosest at the center. To tighten: **branch on $x$ (or $y$)** to split the box, recompute envelopes on each child. Spatial B&B (Lesson 21) is exactly this iterated.

The choice of which variable to branch on is the spatial branching rule; common heuristics are *most violating* and *largest box*.

## 5. RLT (Reformulation-Linearization Technique)

A more aggressive relaxation: multiply pairs of constraints together (linear × linear → bilinear, then McCormick on each). Tightens the LP relaxation at the cost of a quadratic blow-up in size {cite:p}`Sherali1990`.

## References
{cite:p}`McCormick1976,Adjiman1998,Adjiman1998a,Tawarmalani2005,Sherali1990`.